# Exercise 02 — Risk and return: the historical record

MSc Finance · Investments · FHNW · Autumn 2026

A century of daily U.S. market returns: what the tails look like, how badly the normal
distribution describes them, and why the answer changes with the investment horizon.

**How to work with this notebook.** The task text is here and on the exercise sheet. Each
code cell is a stub: the `# TODO` lines are the steps, in order. The setup and data cells
below are complete — run them first and leave them alone.

Run **Runtime → Restart and run all** before you trust any number in here.

**Data.** Kenneth R. French Data Library, *Fama/French 3 Factors (Daily)*, file
`F-F_Research_Data_Factors_daily.csv`, 202512 CRSP vintage. The market return used
throughout is $R = (\text{Mkt-RF}) + \text{RF}$, the total return on the CRSP
value-weighted index, in U.S. dollars. Source and documentation:
[mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from exercise_utils import FHNW, setup_style, describe, save_results
setup_style()

In [ ]:
BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_02/data/F-F_Research_Data_Factors_daily.csv"

# The file carries four lines of documentation on top and a copyright line at the
# bottom; returns are in percent and the date is an integer YYYYMMDD.
ff = pd.read_csv(DATA_URL, skiprows=4, skipfooter=2, engine="python")
ff.columns = ["date", "Mkt-RF", "SMB", "HML", "RF"]
ff["date"] = pd.to_datetime(ff["date"].astype(int).astype(str), format="%Y%m%d")
ff = ff.set_index("date").astype(float) / 100.0

R = (ff["Mkt-RF"] + ff["RF"]).rename("Market")   # total return on the market
rf = ff["RF"].rename("RF")                       # daily risk-free rate

print(f"{len(R)} trading days, {R.index.min().date()} to {R.index.max().date()}")

## Task 1 — The wealth index and the worst that ever happened

Cumulate the daily market return into a wealth index: one dollar invested on the first day
of the sample. The Fama/French series is in U.S. dollars throughout, so every level in this
exercise is a dollar amount and every return is an unhedged USD return. Find the five worst single days and express each one in standard deviations
from the daily mean. Then compute the drawdown series — the distance between the wealth
index and its own running maximum — and report the largest drawdown together with three
dates: the peak it fell from, the trough, and the day the loss was finally recovered.

Draw the wealth index and the drawdown series in one two-panel figure.

*Deliverable: the table of five worst days in σ units, the maximum drawdown with its three
dates, and the two-panel figure.*

In [ ]:
# TODO: cumulate the daily returns into a wealth index W (one USD at the start)
# TODO: the five worst days, in percent and in standard deviations from the mean
# TODO: drawdown = wealth index relative to its running maximum, minus one
# TODO: the maximum drawdown and its peak, trough and recovery dates

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})
# TODO: top panel — the wealth index. Think about the vertical scale before you plot it.
# TODO: bottom panel — the drawdown series
fig.tight_layout()
plt.show()

## Task 2 — How wrong is the normal distribution?

Report the mean, the volatility, the skewness and the excess kurtosis of the daily series.
Then draw the histogram of the daily returns with a fitted normal density on top — **twice,
side by side**: once with a linear density axis and once with a logarithmic one. The two
panels show the same two objects; decide for yourself which of them you would put in a risk
report.

Now put a number on that disagreement. Compute the Value at Risk and the Expected
Shortfall at the 5 % and the 1 % level, each of them twice:

- parametrically, assuming $R \sim N(\hat\mu, \hat\sigma^2)$, and
- empirically, read straight off the 26,151 observations in the sample.

Finally, compute the probability of a day worse than −5 % both ways.

*Deliverable: the two-panel histogram, a table with four tail numbers at each of the two
confidence levels, and the two shortfall probabilities.*

In [ ]:
mu, sd = R.mean(), R.std()
# TODO: mean, volatility, skewness and excess kurtosis of the daily series
fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
grid = np.linspace(-0.20, 0.20, 800)
for ax in axes:
    ...
    # TODO: histogram of the daily returns, and the normal density with the same mu and sd
# TODO: left panel linear, right panel logarithmic
fig.tight_layout()
plt.show()

In [ ]:
for alpha in [0.05, 0.01]:
    z = stats.norm.ppf(alpha)
    # TODO: VaR and ES under the normal assumption (both reported as positive losses)
    # TODO: the same two numbers read directly off the sample
# TODO: probability of a day worse than -5 %, empirically and under normality

## Task 3 — Where the tails come from, and where they go

Fat tails are not a property that returns simply have. They are produced by volatility that
changes over time: a mixture of quiet periods and violent ones has fatter tails than any
single normal distribution, even when every individual period is normal.

First look at the volatility itself. Plot the rolling 252-day standard deviation,
annualised, and mark the full-sample volatility as a horizontal reference. Report the
lowest, the highest and the median value of the rolling series.

Then aggregate. Use the `to_frequency()` function in the next cell to compound the daily
returns to monthly and annual returns, and use `describe()` from `exercise_utils` to report
the number of observations, the annualised mean and volatility, the skewness, the excess
kurtosis and the Sharpe ratio at each of the three frequencies.

Two columns of that table behave completely differently as the horizon lengthens: the
excess kurtosis and the Sharpe ratio. Say which of the two is roughly horizon-invariant,
and why the other is not.

*Deliverable: the rolling-volatility figure with its three numbers, the three-row moments
table, and one sentence contrasting the two columns.*

In [ ]:
# TODO: rolling 252-day standard deviation, annualised
fig, ax = plt.subplots(figsize=(9, 3.6))
# TODO: plot it, with the full-sample annualised volatility as a horizontal line
plt.show()
# TODO: lowest, highest and median of the rolling series

In [ ]:
def to_frequency(x, rule):
    # Compound a daily return series into lower-frequency returns.
    #
    # rule: "ME" month end, "YE" year end, "QE" quarter end (pandas 2.2 and later;
    # the older aliases "M", "A", "Q" are deprecated).
    #
    # Returns compound, they do not add up — hence prod(), never sum().
    return (1 + x).resample(rule).prod() - 1


PERIODS = {"Daily": 252, "Monthly": 12, "Annual": 1}
RULES = {"Daily": None, "Monthly": "ME", "Annual": "YE"}

In [ ]:
for label, rule in RULES.items():
    x = R if rule is None else to_frequency(R, rule)
    y = rf if rule is None else to_frequency(rf, rule)
    # TODO: describe() the market at this frequency, annualising with PERIODS[label]

## Export

Bundle the figures and the tables you produced, in case you want them for the transfer
questions or your own notes.

In [ ]:
# TODO: export the moments and tail tables together with one figure

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Lecture 03 takes the two moments that survived this exercise intact — the mean and the
volatility — and asks how much of your wealth belongs in the risky asset at all.